### NLP - (Natural Language Processing)

- NLP is Most Commanly stands For Natural Language Processing,   A branch of artificial intelligence that helps computers to Read,   Hear, Speak, And Understand Human text And Speech

In [16]:
import re
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sk_text
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 80)


In [17]:
df = pd.read_csv('dataset-tickets-first-200.csv')
df.head()

,subject,body,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Unvorhergesehener Absturz der Datenanalyse-Plattform,"Die Datenanalyse-Plattform brach unerwartet ab, da die Speicheroberfläche zu...","Ich werde Ihnen bei der Lösung des Problems helfen, indem die Datenanalyse-P...",Incident,General Inquiry,low,de,Crash,Technical,Bug,Hardware,Resolution,Outage,Documentation,NaN
1,Customer Support Inquiry,Seeking information on digital strategies that can aid in brand growth and d...,"We offer a variety of digital strategies and services to boost brand growth,...",Request,Customer Service,medium,en,Feedback,Sales,IT,Tech Support,NaN,NaN,NaN,NaN
2,Data Analytics for Investment,I am contacting you to request information on data analytics tools that can ...,I am here to assist you with data analytics tools for investment optimizatio...,Request,Customer Service,medium,en,Technical,Product,Guidance,Documentation,Performance,Feature,NaN,NaN
3,Krankenhaus-Dienstleistung-Problem,Ein Medien-Daten-Sperrverhalten trat aufgrund unerlaubten Zugriffes auf. Ein...,Zurück zur E-Mail-Beschwerde über den Sperrversuch auf Grund unerlaubten Zug...,Incident,Customer Service,high,de,Security,Breach,Login,Maintenance,Incident,Resolution,Feedback,NaN
4,Security,"Dear Customer Support, I am reaching out to inquire about the security proto...","Dear [name], we take the security of medical data very seriously and have im...",Request,Customer Service,medium,en,Security,Customer,Compliance,Breach,Documentation,Guidance,NaN,NaN


In [18]:
# What are we predicting? -> 'queue' : which support team should handle this ticket
df['queue'].value_counts()

queue
Technical Support                  73
Customer Service                   29
Product Support                    23
IT Support                         21
Billing and Payments               21
Returns and Exchanges              12
Sales and Pre-Sales                 8
Service Outages and Maintenance     6
General Inquiry                     4
Human Resources                     3
Name: count, dtype: int64

In [19]:
# Language mix and missing values -- real data always needs a sanity check first
print("Languages:\n", df['language'].value_counts())
print("\nMissing values per column:\n", df[['subject','body','queue','language']].isnull().sum())

Languages:
 language
en    127
de     73
Name: count, dtype: int64

Missing values per column:
 subject     13
body         0
queue        0
language     0
dtype: int64


In [20]:
# Combine subject + body into a single text field (fill missing subject with empty string)
df['subject'] = df['subject'].fillna('')
df['text'] = (df['subject'] + ' ' + df['body']).str.strip()

df[['text', 'queue', 'language']].head(3)

,text,queue,language
0,Unvorhergesehener Absturz der Datenanalyse-Plattform Die Datenanalyse-Plattf...,General Inquiry,de
1,Customer Support Inquiry Seeking information on digital strategies that can ...,Customer Service,en
2,Data Analytics for Investment I am contacting you to request information on ...,Customer Service,en


In [21]:
GERMAN_STOPWORDS = set('''
der die das ein eine einer eines einem einen und oder aber ist sind war waren
ich du er sie es wir ihr auf in an im am zu zum zur von vom mit bei nach für
auf dem den des nicht kein keine nur auch noch schon wie was wann wo wer wie
sich uns euch mich mir dich dir sein ihre ihrer haben hat habe wurde wird
werden können kann könnte muss müssen sollte soll bitte danke vielen dank
'''.split())

ENGLISH_STOPWORDS = set(sk_text.ENGLISH_STOP_WORDS)
ALL_STOPWORDS = ENGLISH_STOPWORDS | GERMAN_STOPWORDS

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)                    # strip any HTML-ish tags/placeholders
    text = re.sub(r'[^a-zäöüß\s]', ' ', text)                # keep letters incl. German, drop digits/punct
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in ALL_STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)
df[['text', 'clean_text']].head(3)

,text,clean_text
0,Unvorhergesehener Absturz der Datenanalyse-Plattform Die Datenanalyse-Plattf...,unvorhergesehener absturz datenanalyse plattform datenanalyse plattform brac...
1,Customer Support Inquiry Seeking information on digital strategies that can ...,customer support inquiry seeking information digital strategies aid brand gr...
2,Data Analytics for Investment I am contacting you to request information on ...,data analytics investment contacting request information data analytics tool...
